## From Material Point to Global System

---

## Level 1: Material Point (UMAT)

At each Gauss point you provide:

$$\texttt{DDSDDE} = \mathfrak{A}_{n+1} = \frac{\partial\Delta\boldsymbol{\sigma}}{\partial\Delta\boldsymbol{\varepsilon}} \quad (6\times6)$$

This lives at a **single point** in space — it has no knowledge of geometry.

---

## Level 2: Element Stiffness Matrix

The strain at a Gauss point is related to nodal displacements through the **strain-displacement matrix** $\mathbf{B}$:

$$\Delta\boldsymbol{\varepsilon} = \mathbf{B}\Delta\mathbf{u}^e$$

where $\mathbf{B}$ encodes the derivatives of shape functions:

$$\mathbf{B} = \begin{bmatrix} \partial N_1/\partial x & 0 & 0 & \partial N_2/\partial x & \cdots \\ 0 & \partial N_1/\partial y & 0 & 0 & \cdots \\ \vdots & & \ddots & & \end{bmatrix}$$

The element stiffness is assembled by integrating over all Gauss points $g$:

$$\mathbf{K}^e = \int_{\Omega^e} \mathbf{B}^T \underbrace{\mathfrak{A}_{n+1}}_{\texttt{DDSDDE}} \mathbf{B} \, dV \approx \sum_g w_g \mathbf{B}_g^T \mathfrak{A}_{n+1}^g \mathbf{B}_g \det(\mathbf{J}_g)$$

where $w_g$ are Gauss weights and $\mathbf{J}_g$ is the Jacobian of the isoparametric mapping.

Dimensions:

$$\underbrace{\mathbf{K}^e}_{(n_{dof}^e \times n_{dof}^e)} = \int \underbrace{\mathbf{B}^T}_{(n_{dof}^e \times 6)} \underbrace{\mathfrak{A}_{n+1}}_{(6\times6)} \underbrace{\mathbf{B}}_{(6\times n_{dof}^e)} dV$$

---

## Level 3: Global Stiffness Assembly

Each element contributes to the global stiffness via the **assembly operator** $\mathbf{L}^e$ (scatter matrix mapping local to global DOFs):

$$\mathbf{K}^{glob} = \bigcup_{e=1}^{N_{elem}} \mathbf{L}^{eT} \mathbf{K}^e \mathbf{L}^e = \bigcup_{e=1}^{N_{elem}} \int_{\Omega^e} \mathbf{B}^T \mathfrak{A}_{n+1} \mathbf{B} \, dV$$

In practice this is just **adding element stiffness entries into the right global DOF positions**.

---

## Level 4: Global Newton-Raphson

Abaqus solves:

$$\mathbf{K}^{glob} \Delta\mathbf{u} = \mathbf{R}^{glob}$$

where the residual is:

$$\mathbf{R}^{glob} = \mathbf{f}^{ext} - \mathbf{f}^{int} = \mathbf{f}^{ext} - \bigcup_e\int_{\Omega^e}\mathbf{B}^T\boldsymbol{\sigma}_{n+1}\,dV$$

Each Newton iteration:

$$\Delta\mathbf{u}^{k+1} = \Delta\mathbf{u}^k + \delta\mathbf{u}^k$$
$$\mathbf{K}^{glob,k}\delta\mathbf{u}^k = \mathbf{R}^{glob,k}$$

---

## The Complete Flow

```
UMAT (each Gauss point)
│
│  input:  DSTRAN (De)
│  solve:  y_{n+1}
│  output: STRESS, STATEV
│          DDSDDE = A_{n+1}  (6x6)
│
▼
B^T * DDSDDE * B              (n_dof^e x n_dof^e)
│
│  integrate over Gauss points
│  weighted sum: sum_g w_g * B_g^T * A_g * B_g * det(J_g)
│
▼
K^e  (element stiffness)
│
│  scatter into global positions
│  via assembly operator L^e
│
▼
K^glob  (global stiffness)
│
│  K^glob * du = R^glob
│  solve linear system (sparse direct or iterative)
│
▼
u_{n+1}  (updated displacement)
│
│  compute new strains: De = B * du
│  call UMAT again at each Gauss point
│
▼
repeat until ||R^glob|| < tolerance
```

---

## Why DDSDDE Quality Propagates to Global Level

| DDSDDE quality | $\mathbf{K}^e$ quality | $\mathbf{K}^{glob}$ quality | Global Newton |
|---|---|---|---|
| Exact consistent | Exact | Exact | Quadratic convergence |
| Elastic only | Too stiff/soft | Wrong | Linear convergence |
| Wrong sign | Indefinite | Indefinite | Divergence |
| Approximate | Approximate | Approximate | Sub-quadratic |

---

## Key Insight

$$\boxed{\mathfrak{A}_{n+1} \text{ at one Gauss point} \xrightarrow{\mathbf{B}^T(\cdot)\mathbf{B}} \mathbf{K}^e \xrightarrow{\text{assembly}} \mathbf{K}^{glob} \xrightarrow{\text{solve}} \delta\mathbf{u}}$$

> DDSDDE is the **only material information** Abaqus uses to build the global stiffness. Everything else — $\mathbf{B}$, Gauss weights, assembly — is purely geometric. A wrong DDSDDE corrupts $\mathbf{K}^{glob}$ at every DOF connected to that Gauss point, degrading convergence globally.

## Full Matrix Forms

---

## DDSDDE — $(6\times6)$

$$\mathfrak{A}_{n+1} = \begin{bmatrix} \frac{\partial\sigma^1}{\partial\varepsilon^1} & \frac{\partial\sigma^1}{\partial\varepsilon^2} & \frac{\partial\sigma^1}{\partial\varepsilon^3} & \frac{\partial\sigma^1}{\partial\varepsilon^4} & \frac{\partial\sigma^1}{\partial\varepsilon^5} & \frac{\partial\sigma^1}{\partial\varepsilon^6} \\ \frac{\partial\sigma^2}{\partial\varepsilon^1} & \frac{\partial\sigma^2}{\partial\varepsilon^2} & \frac{\partial\sigma^2}{\partial\varepsilon^3} & \frac{\partial\sigma^2}{\partial\varepsilon^4} & \frac{\partial\sigma^2}{\partial\varepsilon^5} & \frac{\partial\sigma^2}{\partial\varepsilon^6} \\ \frac{\partial\sigma^3}{\partial\varepsilon^1} & \frac{\partial\sigma^3}{\partial\varepsilon^2} & \frac{\partial\sigma^3}{\partial\varepsilon^3} & \frac{\partial\sigma^3}{\partial\varepsilon^4} & \frac{\partial\sigma^3}{\partial\varepsilon^5} & \frac{\partial\sigma^3}{\partial\varepsilon^6} \\ \frac{\partial\sigma^4}{\partial\varepsilon^1} & \frac{\partial\sigma^4}{\partial\varepsilon^2} & \frac{\partial\sigma^4}{\partial\varepsilon^3} & \frac{\partial\sigma^4}{\partial\varepsilon^4} & \frac{\partial\sigma^4}{\partial\varepsilon^5} & \frac{\partial\sigma^4}{\partial\varepsilon^6} \\ \frac{\partial\sigma^5}{\partial\varepsilon^1} & \frac{\partial\sigma^5}{\partial\varepsilon^2} & \frac{\partial\sigma^5}{\partial\varepsilon^3} & \frac{\partial\sigma^5}{\partial\varepsilon^4} & \frac{\partial\sigma^5}{\partial\varepsilon^5} & \frac{\partial\sigma^5}{\partial\varepsilon^6} \\ \frac{\partial\sigma^6}{\partial\varepsilon^1} & \frac{\partial\sigma^6}{\partial\varepsilon^2} & \frac{\partial\sigma^6}{\partial\varepsilon^3} & \frac{\partial\sigma^6}{\partial\varepsilon^4} & \frac{\partial\sigma^6}{\partial\varepsilon^5} & \frac{\partial\sigma^6}{\partial\varepsilon^6} \end{bmatrix}$$

---

## Strain-Displacement Matrix $\mathbf{B}$ — $(6\times n_{dof}^e)$

For a 3D element with $n$ nodes, each with 3 displacement DOFs, $n_{dof}^e = 3n$:

$$\mathbf{B} = \begin{bmatrix} \frac{\partial N_1}{\partial x} & 0 & 0 & \frac{\partial N_2}{\partial x} & 0 & 0 & \cdots \\ 0 & \frac{\partial N_1}{\partial y} & 0 & 0 & \frac{\partial N_2}{\partial y} & 0 & \cdots \\ 0 & 0 & \frac{\partial N_1}{\partial z} & 0 & 0 & \frac{\partial N_2}{\partial z} & \cdots \\ \frac{\partial N_1}{\partial y} & \frac{\partial N_1}{\partial x} & 0 & \frac{\partial N_2}{\partial y} & \frac{\partial N_2}{\partial x} & 0 & \cdots \\ 0 & \frac{\partial N_1}{\partial z} & \frac{\partial N_1}{\partial y} & 0 & \frac{\partial N_2}{\partial z} & \frac{\partial N_2}{\partial y} & \cdots \\ \frac{\partial N_1}{\partial z} & 0 & \frac{\partial N_1}{\partial x} & \frac{\partial N_2}{\partial z} & 0 & \frac{\partial N_2}{\partial x} & \cdots \end{bmatrix}$$

---

## Element Stiffness — $(n_{dof}^e \times n_{dof}^e)$

$$\mathbf{K}^e = \sum_g w_g \det(\mathbf{J}_g) \cdot \mathbf{B}_g^T \mathfrak{A}_{n+1}^g \mathbf{B}_g$$

Written out explicitly for one Gauss point:

$$\mathbf{B}^T \mathfrak{A}_{n+1} \mathbf{B} = \begin{bmatrix} \frac{\partial N_1}{\partial x} & 0 & 0 & \frac{\partial N_1}{\partial y} & 0 & \frac{\partial N_1}{\partial z} \\ 0 & \frac{\partial N_1}{\partial y} & 0 & \frac{\partial N_1}{\partial x} & \frac{\partial N_1}{\partial z} & 0 \\ 0 & 0 & \frac{\partial N_1}{\partial z} & 0 & \frac{\partial N_1}{\partial y} & \frac{\partial N_1}{\partial x} \\ \vdots & \vdots & \vdots & \vdots & \vdots & \vdots \end{bmatrix} \begin{bmatrix} \frac{\partial\sigma^1}{\partial\varepsilon^1} & \cdots & \frac{\partial\sigma^1}{\partial\varepsilon^6} \\ \vdots & \ddots & \vdots \\ \frac{\partial\sigma^6}{\partial\varepsilon^1} & \cdots & \frac{\partial\sigma^6}{\partial\varepsilon^6} \end{bmatrix} \begin{bmatrix} \frac{\partial N_1}{\partial x} & 0 & 0 & \cdots \\ 0 & \frac{\partial N_1}{\partial y} & 0 & \cdots \\ 0 & 0 & \frac{\partial N_1}{\partial z} & \cdots \\ \frac{\partial N_1}{\partial y} & \frac{\partial N_1}{\partial x} & 0 & \cdots \\ 0 & \frac{\partial N_1}{\partial z} & \frac{\partial N_1}{\partial y} & \cdots \\ \frac{\partial N_1}{\partial z} & 0 & \frac{\partial N_1}{\partial x} & \cdots \end{bmatrix}$$

---

## Global Assembly — $(n_{dof}^{glob} \times n_{dof}^{glob})$

$$\mathbf{K}^{glob} = \bigcup_{e=1}^{N_{elem}} \mathbf{K}^e$$

Concretely for 2 elements sharing nodes:

$$\mathbf{K}^{glob} = \begin{bmatrix} K^1_{11} & K^1_{12} & K^1_{13} & 0 \\ K^1_{21} & K^1_{22}+K^2_{11} & K^1_{23}+K^2_{12} & K^2_{13} \\ K^1_{31} & K^1_{32}+K^2_{21} & K^1_{33}+K^2_{22} & K^2_{23} \\ 0 & K^2_{31} & K^2_{32} & K^2_{33} \end{bmatrix}$$

Shared DOFs **add** — this is the assembly.

---

## Global Newton System

$$\underbrace{\mathbf{K}^{glob}}_{(n_{dof}^{glob}\times n_{dof}^{glob})} \underbrace{\delta\mathbf{u}}_{(n_{dof}^{glob}\times 1)} = \underbrace{\mathbf{R}^{glob}}_{(n_{dof}^{glob}\times 1)}$$

$$\begin{bmatrix} K_{11} & K_{12} & \cdots & K_{1n} \\ K_{21} & K_{22} & \cdots & K_{2n} \\ \vdots & & \ddots & \vdots \\ K_{n1} & K_{n2} & \cdots & K_{nn} \end{bmatrix} \begin{bmatrix} \delta u_1 \\ \delta u_2 \\ \vdots \\ \delta u_n \end{bmatrix} = \begin{bmatrix} f_1^{ext} - f_1^{int} \\ f_2^{ext} - f_2^{int} \\ \vdots \\ f_n^{ext} - f_n^{int} \end{bmatrix}$$

where:

$$f_i^{int} = \bigcup_e \int_{\Omega^e} \mathbf{B}^T \begin{bmatrix}\sigma^1\\\sigma^2\\\sigma^3\\\sigma^4\\\sigma^5\\\sigma^6\end{bmatrix} dV = \bigcup_e \sum_g w_g \det(\mathbf{J}_g) \mathbf{B}_g^T \begin{bmatrix}\sigma^1\\\sigma^2\\\sigma^3\\\sigma^4\\\sigma^5\\\sigma^6\end{bmatrix}_g$$

---

## The Dimension Chain

$$\underbrace{\mathfrak{A}_{n+1}}_{6\times6} \xrightarrow{\mathbf{B}^T_{(n_{dof}^e\times6)}(\cdot)\mathbf{B}_{(6\times n_{dof}^e)}} \underbrace{\mathbf{K}^e}_{n_{dof}^e\times n_{dof}^e} \xrightarrow{\text{assembly}} \underbrace{\mathbf{K}^{glob}}_{n_{dof}^{glob}\times n_{dof}^{glob}} \xrightarrow{\text{solve}} \underbrace{\delta\mathbf{u}}_{n_{dof}^{glob}\times1}$$

> Every entry of $\mathbf{K}^{glob}$ is a **weighted sum of DDSDDE entries**, filtered through shape function derivatives. A wrong DDSDDE entry $\partial\sigma^I/\partial\varepsilon^J$ corrupts every global DOF pair $(i,j)$ whose shape functions have nonzero $\partial N/\partial x_I$ and $\partial N/\partial x_J$ at that Gauss point.